# SynDiff Preprocessing
- hardcoded input (T1) and target (T2) folder names
- therefore, T1c will always be labeled under T2 as the target
- T1 folders vary based on input 
- syndiff also structures data in .mat files

In [ ]:
import os
import shutil
import numpy as np
from pathlib import Path
from PIL import Image
import h5py

# configs
source_split_root = Path('/path/to/split_root')  
syndiff_root = Path('/path/to/syndiff/initial_split')                   
flat_root = Path('/path/to/syndiff/flat_input')        
mat_root = Path('/path/to/syndiff/mat_output')        

subfolders = ["t1n", "t1c", "t2f", "seg_scaled"]              # copied into syndiff split folders
splits = ["train", "validation", "test"]


# possible input conditions hardcoded as T1 for the SynDiff code to work properly 
INPUT_COND_T1 = "flair"           # uses t2f directly  (no overlay created)
# INPUT_COND_T1 = "t1n"           # uses t1n directly  (no overlay created)
# INPUT_COND_T1 = "t1n_seg"       # uses t1n + α·seg_scaled overlay

ALPHA = 0.3                   # only used when t1n_seg (change to 0.1 for lighter overlay)

# T2 (target) is always t1c
TARGET_T2 = "t1c"

# SyNDiff uses "val" not "validation" 
split_label = {"train": "train", "validation": "val", "test": "test"}


### Copy in data into SynDiff folders

In [ ]:
# Copy into SynDiff first
for split in splits:
    src_root = source_split_root / split
    dst_root = syndiff_root / split

    for patient_dir in sorted(p for p in src_root.iterdir() if p.is_dir()):
        dst_patient = dst_root / patient_dir.name
        dst_patient.mkdir(parents=True, exist_ok=True)

        for sub in subfolders:
            src_sub = patient_dir / sub
            dst_sub = dst_patient / sub
            if src_sub.exists():
                shutil.copytree(src_sub, dst_sub, dirs_exist_ok=True)
            else:
                print(f" {split}/{patient_dir.name} — missing {sub}")

        print(f"{split}/{patient_dir.name} done ")

# Create the overlay condition
overlay_folder_name = f"t1n_seg_{int(ALPHA * 10):02d}"   # e.g. "t1_seg_03" or "t1_seg_01"

if INPUT_COND_T1 == "t1n_seg":
    for split in splits:
        print(f" {split}")
        for patient_dir in sorted(p for p in (syndiff_root / split).iterdir() if p.is_dir()):
            t1n_dir = patient_dir / "t1n"
            seg_dir = patient_dir / "seg_scaled"
            out_dir = patient_dir / overlay_folder_name
            out_dir.mkdir(parents=True, exist_ok=True)

            for t1n_path in sorted(t1n_dir.glob("*.png")):
                seg_path = seg_dir / t1n_path.name
                if not seg_path.exists():
                    print(f"  [WARN] Missing seg_scaled: {t1n_path.name} for {patient_dir.name}")
                    continue
                t1n = np.array(Image.open(t1n_path)).astype(np.float32)
                seg = np.array(Image.open(seg_path)).astype(np.float32)
                combined = np.clip(t1n + ALPHA * seg, 0, 255).astype(np.uint8)
                Image.fromarray(combined).save(out_dir / t1n_path.name)

            print(f"{patient_dir.name} done")



### Flatten folders and name according to the T1 and T2 conventions

In [ ]:
# mapping folder to T1 or T2 conditions
t1_source_map = {
    "flair":   "t2f",
    "t1n":     "t1n",
    "t1n_seg": overlay_folder_name,
}
t1_subfolder = t1_source_map[INPUT_COND_T1]

# Creating and flattening the T1 and T2 folders
for split in splits:
    label= split_label[split]

    # SynDiff folder naming convention (data_*_T1)
    t1_out= flat_root / f"data_{label}_T1"
    t2_out= flat_root / f"data_{label}_T2"
    t1_out.mkdir(parents=True, exist_ok=True)
    t2_out.mkdir(parents=True, exist_ok=True)

    for patient_dir in sorted(p for p in (syndiff_root / split).iterdir() if p.is_dir()):
        t1_dir = patient_dir / t1_subfolder
        t2_dir = patient_dir / TARGET_T2

        if not t1_dir.exists():
            print(f" {patient_dir.name} — missing {t1_subfolder}")
            continue
        if not t2_dir.exists():
            print(f" {patient_dir.name} — missing {TARGET_T2}")
            continue

        t1_files = sorted(t1_dir.glob("*.png"))
        t2_files = sorted(t2_dir.glob("*.png"))
        if len(t1_files) != len(t2_files):
            print(f" {patient_dir.name} — T1/T2 count mismatch, using minimum")

        for img_path in t1_files:
            shutil.copy(img_path, t1_out / f"{patient_dir.name}_{img_path.name}")
        for img_path in t2_files:
            shutil.copy(img_path, t2_out / f"{patient_dir.name}_{img_path.name}")

        print(f" [{patient_dir.name} ({len(t1_files)} slices)")



### Create the .mat files

In [ ]:
mat_root.mkdir(parents=True, exist_ok=True)
batch_size = 100    # to speed up processing 


for split in splits:
    label = split_label[split]
    t1_folder = flat_root / f"data_{label}_T1"
    t2_folder = flat_root / f"data_{label}_T2"

    t1_files = sorted(f for f in t1_folder.iterdir() if f.suffix == ".png")
    t2_files = sorted(f for f in t2_folder.iterdir() if f.suffix == ".png")

    if len(t1_files) != len(t2_files):
        raise ValueError(f"Slice count mismatch in {split}: {len(t1_files)} T1 vs {len(t2_files)} T2")

    num_slices   = len(t1_files)
    height, width = np.array(Image.open(t1_files[0])).shape

    t1_mat = mat_root / f"data_{label}_T1.mat"
    t2_mat = mat_root / f"data_{label}_T2.mat"

    with h5py.File(t1_mat, 'w') as f1, h5py.File(t2_mat, 'w') as f2:
        dset1 = f1.create_dataset("data_fs", shape=(num_slices, height, width), dtype=np.float32)
        dset2 = f2.create_dataset("data_fs", shape=(num_slices, height, width), dtype=np.float32)

        for start in range(0, num_slices, batch_size):
            end = min(start + batch_size, num_slices)
            dset1[start:end] = np.stack([np.array(Image.open(f)).astype(np.float32) / 255.0
                                          for f in t1_files[start:end]])
            dset2[start:end] = np.stack([np.array(Image.open(f)).astype(np.float32) / 255.0
                                          for f in t2_files[start:end]])
            print(f"Written slices {start}–{end-1}")

    print(f"{t1_mat.name} + {t2_mat.name}")